In [3]:
import pandas as pd
from sqlalchemy import create_engine, text

# Create SQLite database connection

engine = create_engine("sqlite:///bluestock_mf.db")

# Read datasets

df_fund = pd.read_csv("../data/raw/01_fund_master.csv")
df_aum = pd.read_csv("../data/raw/03_aum_by_fund_house.csv")

df_nav = pd.read_csv("../data/processed/nav_history_clean.csv")
df_perf = pd.read_csv("../data/processed/clean_scheme_performance.csv")
df_txn = pd.read_csv("../data/processed/clean_investor_transactions.csv")

# Load tables into SQLite

df_fund.to_sql(
"dim_fund",
engine,
if_exists="replace",
index=False
)

df_nav.to_sql(
"fact_nav",
engine,
if_exists="replace",
index=False
)

df_txn.to_sql(
"fact_transactions",
engine,
if_exists="replace",
index=False
)

df_perf.to_sql(
"fact_performance",
engine,
if_exists="replace",
index=False
)

df_aum.to_sql(
"fact_aum",
engine,
if_exists="replace",
index=False
)

print("Data loaded successfully!")

# Verify row counts

with engine.connect() as conn:


    print("\n===== ROW COUNT VALIDATION =====")

    print(
        "dim_fund:",
        len(df_fund),
        conn.execute(
            text("SELECT COUNT(*) FROM dim_fund")
        ).scalar()
    )

    print(
        "fact_nav:",
        len(df_nav),
        conn.execute(
            text("SELECT COUNT(*) FROM fact_nav")
        ).scalar()
    )

    print(
        "fact_transactions:",
        len(df_txn),
        conn.execute(
            text("SELECT COUNT(*) FROM fact_transactions")
        ).scalar()
    )

    print(
        "fact_performance:",
        len(df_perf),
        conn.execute(
            text("SELECT COUNT(*) FROM fact_performance")
        ).scalar()
    )

    print(
        "fact_aum:",
        len(df_aum),
        conn.execute(
            text("SELECT COUNT(*) FROM fact_aum")
        ).scalar()
    )


    print("\nSQLite database created: bluestock_mf.db")


Data loaded successfully!

===== ROW COUNT VALIDATION =====
dim_fund: 40 40
fact_nav: 46000 46000
fact_transactions: 32778 32778
fact_performance: 40 40
fact_aum: 90 90

SQLite database created: bluestock_mf.db


In [5]:
import pandas as pd
from sqlalchemy import create_engine

# -----------------------------
# 1. CREATE SQLITE CONNECTION
# -----------------------------
engine = create_engine("sqlite:///mutual_funds.db")

# -----------------------------
# 2. FILE PATHS (EDIT if needed)
# -----------------------------
files = {
    "dim_fund": "../data/raw/01_fund_master.csv",
    "fact_aum": "../data/raw/03_aum_by_fund_house.csv",
    "fact_nav": "../data/processed/nav_history_clean.csv",
    "fact_transactions": "../data/processed/clean_investor_transactions.csv",
    "fact_performance": "../data/processed/clean_scheme_performance.csv",
    "fact_monthly_sip": "../data/raw/04_monthly_sip_inflows.csv",
    "fact_category_inflows": "../data/raw/05_category_inflows.csv",
    "fact_industry_folio": "../data/raw/06_industry_folio_count.csv",
    "fact_portfolio_holdings": "../data/raw/09_portfolio_holdings.csv",
    "dim_benchmark": "../data/raw/10_benchmark_indices.csv"
}

# -----------------------------
# 3. LOAD + WRITE TO SQLITE + VERIFY
# -----------------------------
for table_name, file_path in files.items():
    df = pd.read_csv(file_path)

    print(f"\nLoading {table_name}...")
    print("CSV rows:", len(df))

    df.to_sql(table_name, engine, if_exists="replace", index=False)

    # verify in DB
    db_count = pd.read_sql(f"SELECT COUNT(*) as cnt FROM {table_name}", engine)['cnt'][0]
    print("SQLite rows:", db_count)

    assert len(df) == db_count, f"Row mismatch in {table_name}!"

print("\n✅ All tables loaded and verified successfully!")


Loading dim_fund...
CSV rows: 40
SQLite rows: 40

Loading fact_aum...
CSV rows: 90
SQLite rows: 90

Loading fact_nav...
CSV rows: 46000
SQLite rows: 46000

Loading fact_transactions...
CSV rows: 32778
SQLite rows: 32778

Loading fact_performance...
CSV rows: 40
SQLite rows: 40

Loading fact_monthly_sip...
CSV rows: 48
SQLite rows: 48

Loading fact_category_inflows...
CSV rows: 144
SQLite rows: 144

Loading fact_industry_folio...
CSV rows: 21
SQLite rows: 21

Loading fact_portfolio_holdings...
CSV rows: 322
SQLite rows: 322

Loading dim_benchmark...
CSV rows: 8050
SQLite rows: 8050

✅ All tables loaded and verified successfully!


In [7]:
import pandas as pd

files = {
    "dim_fund": "../data/raw/01_fund_master.csv",
    "fact_aum": "../data/raw/03_aum_by_fund_house.csv",
    "fact_nav": "../data/processed/nav_history_clean.csv",
    "fact_transactions": "../data/processed/clean_investor_transactions.csv",
    "fact_performance": "../data/processed/clean_scheme_performance.csv",
    "fact_monthly_sip": "../data/raw/04_monthly_sip_inflows.csv",
    "fact_category_inflows": "../data/raw/05_category_inflows.csv",
    "fact_industry_folio": "../data/raw/06_industry_folio_count.csv",
    "fact_portfolio_holdings": "../data/raw/09_portfolio_holdings.csv",
    "dim_benchmark": "../data/raw/10_benchmark_indices.csv"
}

for table_name, path in files.items():
    try:
        df = pd.read_csv(path)
        print("\n" + "="*60)
        print(f"TABLE: {table_name}")
        print("COLUMNS:")
        print(df.columns.tolist())
        print("ROWS:", df.shape[0])
    except Exception as e:
        print(f"\n❌ Error reading {table_name}: {e}")


TABLE: dim_fund
COLUMNS:
['amfi_code', 'fund_house', 'scheme_name', 'category', 'sub_category', 'plan', 'launch_date', 'benchmark', 'expense_ratio_pct', 'exit_load_pct', 'min_sip_amount', 'min_lumpsum_amount', 'fund_manager', 'risk_category', 'sebi_category_code']
ROWS: 40

TABLE: fact_aum
COLUMNS:
['date', 'fund_house', 'aum_lakh_crore', 'aum_crore', 'num_schemes']
ROWS: 90

TABLE: fact_nav
COLUMNS:
['amfi_code', 'date', 'nav']
ROWS: 46000

TABLE: fact_transactions
COLUMNS:
['investor_id', 'transaction_date', 'amfi_code', 'transaction_type', 'amount_inr', 'state', 'city', 'city_tier', 'age_group', 'gender', 'annual_income_lakh', 'payment_mode', 'kyc_status']
ROWS: 32778

TABLE: fact_performance
COLUMNS:
['amfi_code', 'scheme_name', 'fund_house', 'category', 'plan', 'return_1yr_pct', 'return_3yr_pct', 'return_5yr_pct', 'benchmark_3yr_pct', 'alpha', 'beta', 'sharpe_ratio', 'sortino_ratio', 'std_dev_ann_pct', 'max_drawdown_pct', 'aum_crore', 'expense_ratio_pct', 'morningstar_rating', 'r

In [8]:
import pandas as pd
from sqlalchemy import create_engine

engine = create_engine("sqlite:///mutual_funds.db")

# -----------------------------
# 1. TOP SIP INFLOW MONTHS
# -----------------------------
q1 = """
SELECT month, sip_inflow_crore
FROM fact_monthly_sip
ORDER BY sip_inflow_crore DESC
LIMIT 5;
"""

# -----------------------------
# 2. SIP YOY GROWTH TREND
# -----------------------------
q2 = """
SELECT month, yoy_growth_pct
FROM fact_monthly_sip
ORDER BY month;
"""

# -----------------------------
# 3. CATEGORY WISE INFLOWS
# -----------------------------
q3 = """
SELECT category, SUM(net_inflow_crore) AS total_inflow
FROM fact_category_inflows
GROUP BY category
ORDER BY total_inflow DESC;
"""

# -----------------------------
# 4. MONTHLY CATEGORY INFLOW TREND
# -----------------------------
q4 = """
SELECT month, category, net_inflow_crore
FROM fact_category_inflows
ORDER BY month;
"""

# -----------------------------
# 5. INDUSTRY FOLIO BREAKDOWN (LATEST AVAILABLE)
# -----------------------------
q5 = """
SELECT *
FROM fact_industry_folio
ORDER BY month DESC
LIMIT 1;
"""

# -----------------------------
# 6. TOP STOCK HOLDINGS BY WEIGHT
# -----------------------------
q6 = """
SELECT stock_name, SUM(weight_pct) AS total_weight
FROM fact_portfolio_holdings
GROUP BY stock_name
ORDER BY total_weight DESC
LIMIT 10;
"""

# -----------------------------
# 7. TOP SECTORS IN PORTFOLIO
# -----------------------------
q7 = """
SELECT sector, SUM(weight_pct) AS total_weight
FROM fact_portfolio_holdings
GROUP BY sector
ORDER BY total_weight DESC;
"""

# -----------------------------
# 8. BENCHMARK INDEX AVERAGE VALUE
# -----------------------------
q8 = """
SELECT index_name, AVG(close_value) AS avg_close
FROM dim_benchmark
GROUP BY index_name;
"""

# -----------------------------
# 9. NAV TREND BY FUND
# -----------------------------
q9 = """
SELECT amfi_code, AVG(nav) AS avg_nav
FROM fact_nav
GROUP BY amfi_code
ORDER BY avg_nav DESC;
"""

# -----------------------------
# 10. TRANSACTIONS BY TYPE
# -----------------------------
q10 = """
SELECT transaction_type, COUNT(*) AS total_transactions
FROM fact_transactions
GROUP BY transaction_type
ORDER BY total_transactions DESC;
"""

queries = [q1,q2,q3,q4,q5,q6,q7,q8,q9,q10]

for i, q in enumerate(queries, 1):
    print(f"\n--- QUERY {i} ---")
    display(pd.read_sql(q, engine))


--- QUERY 1 ---


,month,sip_inflow_crore
0,2025-12,31002
1,2025-11,30200
2,2025-10,29529
3,2025-09,29361
4,2025-07,28464



--- QUERY 2 ---


,month,yoy_growth_pct
0,2022-01,NaN
1,2022-02,NaN
2,2022-03,NaN
3,2022-04,NaN
4,2022-05,NaN
5,2022-06,NaN
6,2022-07,NaN
7,2022-08,NaN
8,2022-09,NaN
9,2022-10,NaN



--- QUERY 3 ---


...

,index_name,avg_close
0,BSE_SMALLCAP,39375.028357
1,CRISIL_GILT,1779.534591
2,CRISIL_LIQUID,2639.926574
3,NIFTY100,17186.437626
4,NIFTY50,22089.362887
5,NIFTY500,23316.830496
6,NIFTY_MIDCAP150,22076.530739



--- QUERY 9 ---


,amfi_code,avg_nav
0,120844,3705.876331
1,125497,797.454477
2,100016,566.191221
3,149322,439.176645
4,101206,435.975472
5,101208,358.390923
6,120841,355.078750
7,120507,334.083952
8,120505,283.999240
9,120506,274.320937



--- QUERY 10 ---


,transaction_type,total_transactions
0,SIP,19716
1,Lumpsum,8095
2,Redemption,4967


In [9]:
import pandas as pd
from sqlalchemy import create_engine

engine = create_engine("sqlite:///mutual_funds.db")

tables = [
    "fact_monthly_sip",
    "fact_category_inflows",
    "fact_industry_folio",
    "fact_portfolio_holdings",
    "dim_benchmark",
    "fact_nav",
    "fact_transactions",
    "fact_performance",
    "dim_fund",
    "fact_aum"
]

md = "# Mutual Fund Data Dictionary\n\n"

for t in tables:
    df = pd.read_sql(f"SELECT * FROM {t} LIMIT 5", engine)

    md += f"## {t}\n\n"
    md += "| Column | Data Type | Sample Values |\n"
    md += "|--------|----------|--------------|\n"

    for col in df.columns:
        dtype = str(df[col].dtype)
        sample_vals = df[col].dropna().unique()[:3]
        sample = ", ".join(map(str, sample_vals))
        md += f"| {col} | {dtype} | {sample} |\n"

    md += "\n"

with open("data_dictionary.md", "w", encoding="utf-8") as f:
    f.write(md)

print("✅ data_dictionary.md created")

✅ data_dictionary.md created
